In [ ]:
# ============================================================
# OPENLENS:
# TEXTE BEREINIGEN UND IN CHUNKS AUFTEILEN
# ============================================================
#
# Dieser Code:
# - erkennt die OpenLens-Ordner automatisch
# - lädt den Status der PDF-Textextraktion
# - verarbeitet nur erfolgreich extrahierte Texte
# - bereinigt die Texte vorsichtig
# - erhält Seiteninformationen
# - zerlegt die Dokumente in überlappende Textabschnitte
# - ergänzt Dokument-ID, Titel, Quelle und Seitennummern
# - speichert bereinigte Volltexte
# - speichert alle Chunks als JSONL und CSV
#
# Es werden noch KEINE Embeddings erzeugt.
# ============================================================

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd
from IPython.display import display


# ============================================================
# 1. EINSTELLUNGEN
# ============================================================

# Zielgröße eines Chunks in Zeichen.
# 4.000 Zeichen entsprechen grob 700 bis 1.000 Tokens,
# abhängig von Sprache und Textart.
CHUNK_GROESSE_ZEICHEN = 4000

# Überlappung zwischen zwei aufeinanderfolgenden Chunks.
CHUNK_UEBERLAPPUNG_ZEICHEN = 600

# Sehr kleine Reststücke werden möglichst mit dem vorherigen
# Chunk verbunden.
MIN_CHUNK_GROESSE_ZEICHEN = 500

# Nach jeweils so vielen Dokumenten wird zwischengespeichert.
SAVE_EVERY = 20

# Bereits vorhandene Ausgabedateien werden beim Lauf neu aufgebaut.
# Die ursprünglichen extrahierten Texte bleiben unverändert.
AUSGABEDATEIEN_NEU_ERSTELLEN = True


# ============================================================
# 2. PROJEKTORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

if ARBEITSORDNER.name.lower() == "datenbank":
    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENBANKORDNER = ARBEITSORDNER

elif ARBEITSORDNER.name.lower() in {
    "texte",
    "dokumente",
    "bereinigte_texte",
    "chunks",
}:
    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENBANKORDNER = PROJEKTORDNER / "Datenbank"

elif (ARBEITSORDNER / "Datenbank").is_dir():
    PROJEKTORDNER = ARBEITSORDNER
    DATENBANKORDNER = PROJEKTORDNER / "Datenbank"

else:
    raise FileNotFoundError(
        "\nDer OpenLens-Projektordner konnte nicht erkannt werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}\n\n"
        "Erwartet wurde einer dieser Ordner:\n"
        "- OpenLens\n"
        "- OpenLens/Datenbank\n"
        "- OpenLens/Texte\n"
        "- OpenLens/Dokumente"
    )


# ============================================================
# 3. ORDNER UND DATEIEN FESTLEGEN
# ============================================================

TEXTORDNER = PROJEKTORDNER / "Texte"
BEREINIGTE_TEXTORDNER = PROJEKTORDNER / "Bereinigte_Texte"
CHUNK_ORDNER = PROJEKTORDNER / "Chunks"

BEREINIGTE_TEXTORDNER.mkdir(
    parents=True,
    exist_ok=True,
)

CHUNK_ORDNER.mkdir(
    parents=True,
    exist_ok=True,
)

TEXT_STATUS_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_text_extraction_status.csv"
)

DOWNLOAD_STATUS_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_download_status.csv"
)

DOWNLOAD_QUEUE_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_download_queue_all.csv"
)

CHUNKS_JSONL = (
    CHUNK_ORDNER
    / "fragdenstaat_chunks.jsonl"
)

CHUNKS_CSV = (
    CHUNK_ORDNER
    / "fragdenstaat_chunks.csv"
)

DOKUMENT_STATUS_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_chunking_status.csv"
)

DOKUMENT_STATUS_JSONL = (
    DATENBANKORDNER
    / "fragdenstaat_chunking_status.jsonl"
)

CHUNKING_FEHLER_CSV = (
    DATENBANKORDNER
    / "fragdenstaat_chunking_errors.csv"
)


print("=" * 80)
print("OPENLENS: TEXTE BEREINIGEN UND CHUNKS ERZEUGEN")
print("=" * 80)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nExtrahierte Texte:")
print(TEXTORDNER)

print("\nBereinigte Texte:")
print(BEREINIGTE_TEXTORDNER)

print("\nChunk-Ordner:")
print(CHUNK_ORDNER)


# ============================================================
# 4. NOTWENDIGE DATEIEN PRÜFEN
# ============================================================

if not TEXT_STATUS_CSV.exists():
    raise FileNotFoundError(
        "\nDer Textextraktionsstatus wurde nicht gefunden:\n"
        f"{TEXT_STATUS_CSV}\n\n"
        "Führe zuerst die PDF-Textextraktion aus."
    )

if not TEXTORDNER.exists():
    raise FileNotFoundError(
        "\nDer Ordner mit den extrahierten Texten fehlt:\n"
        f"{TEXTORDNER}"
    )


# ============================================================
# 5. TEXTexTRAKTIONSSTATUS LADEN
# ============================================================

status_df = pd.read_csv(
    TEXT_STATUS_CSV,
    encoding="utf-8-sig",
    low_memory=False,
)

benoetigte_statusspalten = [
    "document_id",
    "pdf_name",
    "pdf_path",
    "text_path",
    "text_status",
]

fehlende_statusspalten = [
    spalte
    for spalte in benoetigte_statusspalten
    if spalte not in status_df.columns
]

if fehlende_statusspalten:
    raise KeyError(
        "\nIm Textextraktionsstatus fehlen notwendige Spalten:\n"
        + "\n".join(
            f"- {spalte}"
            for spalte in fehlende_statusspalten
        )
    )


# Nur erfolgreich extrahierte Texte verwenden.
arbeits_df = status_df[
    status_df["text_status"].eq("text_extracted")
].copy()

arbeits_df["document_id"] = pd.to_numeric(
    arbeits_df["document_id"],
    errors="coerce",
)

arbeits_df = arbeits_df[
    arbeits_df["document_id"].notna()
].copy()

arbeits_df["document_id"] = (
    arbeits_df["document_id"]
    .astype("int64")
)

arbeits_df = (
    arbeits_df
    .drop_duplicates(
        subset=["document_id"],
        keep="last",
    )
    .reset_index(drop=True)
)


print("\nErfolgreich extrahierte Dokumente:")
print(f"{len(arbeits_df):,}")


# ============================================================
# 6. OPTIONALE METADATEN LADEN
# ============================================================

metadaten_df = pd.DataFrame()

if DOWNLOAD_STATUS_CSV.exists():

    metadaten_df = pd.read_csv(
        DOWNLOAD_STATUS_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    print("\nMetadatenquelle:")
    print(DOWNLOAD_STATUS_CSV)

elif DOWNLOAD_QUEUE_CSV.exists():

    metadaten_df = pd.read_csv(
        DOWNLOAD_QUEUE_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    print("\nMetadatenquelle:")
    print(DOWNLOAD_QUEUE_CSV)

else:

    print(
        "\nKeine zusätzliche Download-Metadatendatei gefunden. "
        "Titel und Quellen werden soweit möglich aus dem "
        "Textextraktionsstatus übernommen."
    )


if not metadaten_df.empty and "id" in metadaten_df.columns:

    metadaten_df["id"] = pd.to_numeric(
        metadaten_df["id"],
        errors="coerce",
    )

    metadaten_df = metadaten_df[
        metadaten_df["id"].notna()
    ].copy()

    metadaten_df["id"] = (
        metadaten_df["id"]
        .astype("int64")
    )

    verfuegbare_metadaten_spalten = [
        spalte
        for spalte in [
            "id",
            "title",
            "site_url",
            "file_url",
            "num_pages",
            "last_modified_at",
            "publicbody",
            "foirequest",
            "uid",
        ]
        if spalte in metadaten_df.columns
    ]

    metadaten_df = (
        metadaten_df[verfuegbare_metadaten_spalten]
        .drop_duplicates(
            subset=["id"],
            keep="last",
        )
        .rename(
            columns={
                "id": "document_id",
                "num_pages": "metadata_num_pages",
            }
        )
    )

    arbeits_df = arbeits_df.merge(
        metadaten_df,
        on="document_id",
        how="left",
    )


# ============================================================
# 7. HILFSFUNKTIONEN ZUR TEXTBEREINIGUNG
# ============================================================

SEITEN_MARKER_PATTERN = re.compile(
    r"(?m)^===== SEITE\s+(\d+)\s+=====\s*$"
)


def normalisiere_zeilenumbrueche(text: str) -> str:
    """
    Vereinheitlicht Zeilenumbrüche und entfernt Steuerzeichen.
    """

    text = str(text)

    text = text.replace(
        "\ufeff",
        "",
    )

    text = text.replace(
        "\x00",
        "",
    )

    text = text.replace(
        "\r\n",
        "\n",
    )

    text = text.replace(
        "\r",
        "\n",
    )

    return text


def repariere_silbentrennung(text: str) -> str:
    """
    Verbindet Wörter, die am Zeilenende mit Bindestrich
    getrennt wurden.

    Beispiel:
    Informations-
    freiheit

    wird zu:
    Informationsfreiheit
    """

    return re.sub(
        r"(?<=[A-Za-zÄÖÜäöüß])-\n"
        r"(?=[a-zäöüß])",
        "",
        text,
    )


def bereinige_seitentext(text: str) -> str:
    """
    Bereinigt den Text einer einzelnen Seite vorsichtig.
    """

    text = normalisiere_zeilenumbrueche(
        text
    )

    text = repariere_silbentrennung(
        text
    )

    # Leerzeichen und Tabs innerhalb einer Zeile normalisieren.
    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    # Leerzeichen unmittelbar vor Satzzeichen entfernen.
    text = re.sub(
        r"\s+([,.;:!?])",
        r"\1",
        text,
    )

    # Mehrere Leerzeilen auf höchstens zwei reduzieren.
    text = re.sub(
        r"\n[ \t]*\n(?:[ \t]*\n)+",
        "\n\n",
        text,
    )

    # Zeilen, die offenbar nur aus einer Seitenzahl bestehen,
    # entfernen. Längere Zahlen oder Aktenzeichen bleiben erhalten.
    text = re.sub(
        r"(?m)^[ \t]*-?[ \t]*\d{1,4}[ \t]*-?[ \t]*$",
        "",
        text,
    )

    # Erneut übermäßige Leerzeilen reduzieren.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


def dokument_in_seiten_aufteilen(
    text: str,
) -> list[dict]:
    """
    Teilt einen extrahierten Text anhand der eingefügten
    Seitenmarker auf.

    Rückgabe:
    [
        {"page_number": 1, "text": "..."},
        {"page_number": 2, "text": "..."}
    ]
    """

    text = normalisiere_zeilenumbrueche(
        text
    )

    treffer = list(
        SEITEN_MARKER_PATTERN.finditer(text)
    )

    if not treffer:

        bereinigt = bereinige_seitentext(
            text
        )

        return [
            {
                "page_number": 1,
                "text": bereinigt,
            }
        ]


    seiten = []

    for position, match in enumerate(treffer):

        seiten_nummer = int(
            match.group(1)
        )

        start = match.end()

        if position + 1 < len(treffer):
            ende = treffer[position + 1].start()
        else:
            ende = len(text)

        seiten_text = text[
            start:ende
        ]

        seiten_text = bereinige_seitentext(
            seiten_text
        )

        seiten.append(
            {
                "page_number": seiten_nummer,
                "text": seiten_text,
            }
        )

    return seiten


def absatzgrenzen(text: str) -> list[int]:
    """
    Findet sinnvolle mögliche Trennstellen in einem Text.
    """

    grenzen = []

    for match in re.finditer(
        r"\n\n+",
        text,
    ):
        grenzen.append(
            match.end()
        )

    for match in re.finditer(
        r"(?<=[.!?])\s+",
        text,
    ):
        grenzen.append(
            match.end()
        )

    return sorted(
        set(grenzen)
    )


def beste_trennstelle(
    text: str,
    start: int,
    ziel_ende: int,
    minimales_ende: int,
) -> int:
    """
    Sucht vor dem Zielende eine möglichst natürliche
    Trennstelle an Absatz- oder Satzgrenzen.
    """

    teiltext = text[
        start:ziel_ende
    ]

    grenzen = absatzgrenzen(
        teiltext
    )

    gueltige_grenzen = [
        start + grenze
        for grenze in grenzen
        if start + grenze >= minimales_ende
    ]

    if gueltige_grenzen:
        return max(
            gueltige_grenzen
        )

    return ziel_ende


def seiten_fuer_textposition(
    seitenbereiche: list[dict],
    start_position: int,
    end_position: int,
) -> tuple[Optional[int], Optional[int]]:
    """
    Ermittelt die erste und letzte Seite eines Chunks.
    """

    betroffene_seiten = []

    for bereich in seitenbereiche:

        if (
            bereich["end"] > start_position
            and bereich["start"] < end_position
        ):
            betroffene_seiten.append(
                bereich["page_number"]
            )

    if not betroffene_seiten:
        return None, None

    return (
        min(betroffene_seiten),
        max(betroffene_seiten),
    )


def chunks_erzeugen(
    text: str,
    seitenbereiche: list[dict],
) -> list[dict]:
    """
    Zerlegt einen langen Text in überlappende Chunks.
    """

    text = text.strip()

    if not text:
        return []


    if len(text) <= CHUNK_GROESSE_ZEICHEN:

        seite_start, seite_ende = (
            seiten_fuer_textposition(
                seitenbereiche,
                0,
                len(text),
            )
        )

        return [
            {
                "chunk_text": text,
                "character_start": 0,
                "character_end": len(text),
                "page_start": seite_start,
                "page_end": seite_ende,
            }
        ]


    chunks = []
    start = 0
    textlaenge = len(text)

    while start < textlaenge:

        ziel_ende = min(
            start + CHUNK_GROESSE_ZEICHEN,
            textlaenge,
        )

        minimales_ende = min(
            start + int(
                CHUNK_GROESSE_ZEICHEN * 0.65
            ),
            ziel_ende,
        )

        if ziel_ende < textlaenge:

            ende = beste_trennstelle(
                text=text,
                start=start,
                ziel_ende=ziel_ende,
                minimales_ende=minimales_ende,
            )

        else:

            ende = textlaenge


        chunk_text = text[
            start:ende
        ].strip()


        if chunk_text:

            seite_start, seite_ende = (
                seiten_fuer_textposition(
                    seitenbereiche,
                    start,
                    ende,
                )
            )

            chunks.append(
                {
                    "chunk_text": chunk_text,
                    "character_start": start,
                    "character_end": ende,
                    "page_start": seite_start,
                    "page_end": seite_ende,
                }
            )


        if ende >= textlaenge:
            break


        neuer_start = max(
            0,
            ende - CHUNK_UEBERLAPPUNG_ZEICHEN,
        )

        # Verhindert eine Endlosschleife, falls aus irgendeinem
        # Grund keine Vorwärtsbewegung erfolgt.
        if neuer_start <= start:
            neuer_start = ende

        start = neuer_start


    # Sehr kleinen letzten Chunk mit dem vorherigen verbinden.
    if (
        len(chunks) >= 2
        and len(chunks[-1]["chunk_text"])
        < MIN_CHUNK_GROESSE_ZEICHEN
    ):

        letzter_chunk = chunks.pop()
        vorheriger_chunk = chunks[-1]

        kombiniert = (
            vorheriger_chunk["chunk_text"].rstrip()
            + "\n\n"
            + letzter_chunk["chunk_text"].lstrip()
        )

        vorheriger_chunk["chunk_text"] = kombiniert
        vorheriger_chunk["character_end"] = (
            letzter_chunk["character_end"]
        )
        vorheriger_chunk["page_end"] = (
            letzter_chunk["page_end"]
        )


    return chunks


def wert_oder_leer(
    row: pd.Series,
    spalte: str,
):
    """
    Gibt einen sauberen Wert oder einen leeren String zurück.
    """

    if spalte not in row.index:
        return ""

    wert = row.get(spalte)

    if pd.isna(wert):
        return ""

    return wert


# ============================================================
# 8. AUSGABEDATEIEN VORBEREITEN
# ============================================================

if AUSGABEDATEIEN_NEU_ERSTELLEN:

    for datei in [
        CHUNKS_JSONL,
        CHUNKS_CSV,
        DOKUMENT_STATUS_CSV,
        DOKUMENT_STATUS_JSONL,
        CHUNKING_FEHLER_CSV,
    ]:
        if datei.exists():
            datei.unlink()


# ============================================================
# 9. SPEICHERFUNKTION
# ============================================================

def ergebnisse_speichern(
    chunks: list[dict],
    dokument_status: list[dict],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Speichert Chunks und Dokumentstatus.
    """

    chunks_df = pd.DataFrame(
        chunks
    )

    dokument_status_df = pd.DataFrame(
        dokument_status
    )


    if not chunks_df.empty:

        chunks_df = (
            chunks_df
            .drop_duplicates(
                subset=["chunk_id"],
                keep="last",
            )
            .sort_values(
                by=[
                    "document_id",
                    "chunk_index",
                ]
            )
            .reset_index(drop=True)
        )

        chunks_df.to_json(
            CHUNKS_JSONL,
            orient="records",
            lines=True,
            force_ascii=False,
        )

        chunks_df.to_csv(
            CHUNKS_CSV,
            index=False,
            encoding="utf-8-sig",
        )


    if not dokument_status_df.empty:

        dokument_status_df = (
            dokument_status_df
            .drop_duplicates(
                subset=["document_id"],
                keep="last",
            )
            .sort_values(
                by="document_id"
            )
            .reset_index(drop=True)
        )

        dokument_status_df.to_csv(
            DOKUMENT_STATUS_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        dokument_status_df.to_json(
            DOKUMENT_STATUS_JSONL,
            orient="records",
            lines=True,
            force_ascii=False,
        )

        fehler_df = dokument_status_df[
            dokument_status_df["chunking_status"]
            .eq("failed")
        ].copy()

        fehler_df.to_csv(
            CHUNKING_FEHLER_CSV,
            index=False,
            encoding="utf-8-sig",
        )


    return (
        chunks_df,
        dokument_status_df,
    )


# ============================================================
# 10. DOKUMENTE VERARBEITEN
# ============================================================

alle_chunks = []
alle_dokument_status = []

erfolgreiche_dokumente = 0
fehlgeschlagene_dokumente = 0
leere_dokumente = 0


try:

    for position, row in arbeits_df.iterrows():

        laufende_nummer = position + 1
        dokument_id = int(
            row["document_id"]
        )

        text_pfad = Path(
            str(row["text_path"])
        )

        if not text_pfad.is_absolute():
            text_pfad = (
                TEXTORDNER
                / text_pfad.name
            )

        pdf_name = str(
            row.get(
                "pdf_name",
                "",
            )
        )

        titel = str(
            wert_oder_leer(
                row,
                "title",
            )
        ).strip()

        if not titel:
            titel = Path(
                pdf_name
            ).stem

        print(
            f"[{laufende_nummer}/{len(arbeits_df)}] "
            f"ID {dokument_id}: {titel[:70]}",
            end=" ... ",
            flush=True,
        )


        status_eintrag = {
            "document_id": dokument_id,
            "title": titel,
            "pdf_name": pdf_name,
            "source_text_path": str(text_pfad),
            "clean_text_path": "",
            "characters_original": 0,
            "characters_clean": 0,
            "num_pages_detected": 0,
            "num_chunks": 0,
            "chunking_status": "processing",
            "chunking_error": "",
            "processed_at": "",
        }


        try:

            if not text_pfad.exists():
                raise FileNotFoundError(
                    f"Textdatei nicht gefunden: {text_pfad}"
                )


            rohtext = text_pfad.read_text(
                encoding="utf-8",
                errors="replace",
            )

            status_eintrag["characters_original"] = len(
                rohtext
            )


            seiten = dokument_in_seiten_aufteilen(
                rohtext
            )

            status_eintrag["num_pages_detected"] = len(
                seiten
            )


            # Seiten wieder zu einem bereinigten Gesamtdokument
            # verbinden und gleichzeitig Positionsbereiche merken.
            textteile = []
            seitenbereiche = []
            aktuelle_position = 0


            for seite in seiten:

                seiten_text = str(
                    seite["text"]
                ).strip()

                if not seiten_text:
                    continue


                if textteile:
                    trenner = "\n\n"
                    textteile.append(
                        trenner
                    )
                    aktuelle_position += len(
                        trenner
                    )


                seiten_start = aktuelle_position

                textteile.append(
                    seiten_text
                )

                aktuelle_position += len(
                    seiten_text
                )

                seitenbereiche.append(
                    {
                        "page_number": int(
                            seite["page_number"]
                        ),
                        "start": seiten_start,
                        "end": aktuelle_position,
                    }
                )


            bereinigter_text = "".join(
                textteile
            ).strip()

            status_eintrag["characters_clean"] = len(
                bereinigter_text
            )


            if not bereinigter_text:

                status_eintrag["chunking_status"] = (
                    "empty_text"
                )

                status_eintrag["processed_at"] = (
                    datetime.now(
                        timezone.utc
                    ).isoformat()
                )

                alle_dokument_status.append(
                    status_eintrag
                )

                leere_dokumente += 1
                print("LEERER TEXT")
                continue


            bereinigter_text_pfad = (
                BEREINIGTE_TEXTORDNER
                / f"{dokument_id}_{text_pfad.stem}.txt"
            )

            bereinigter_text_pfad.write_text(
                bereinigter_text,
                encoding="utf-8",
            )

            status_eintrag["clean_text_path"] = str(
                bereinigter_text_pfad.resolve()
            )


            dokument_chunks = chunks_erzeugen(
                text=bereinigter_text,
                seitenbereiche=seitenbereiche,
            )


            for chunk_index, chunk in enumerate(
                dokument_chunks
            ):

                chunk_id = (
                    f"{dokument_id}_"
                    f"{chunk_index:05d}"
                )

                chunk_text = chunk[
                    "chunk_text"
                ]

                chunk_eintrag = {
                    "chunk_id": chunk_id,
                    "document_id": dokument_id,
                    "chunk_index": chunk_index,
                    "title": titel,
                    "pdf_name": pdf_name,
                    "page_start": chunk[
                        "page_start"
                    ],
                    "page_end": chunk[
                        "page_end"
                    ],
                    "character_start": chunk[
                        "character_start"
                    ],
                    "character_end": chunk[
                        "character_end"
                    ],
                    "character_count": len(
                        chunk_text
                    ),
                    "word_count": len(
                        re.findall(
                            r"\b\w+\b",
                            chunk_text,
                            flags=re.UNICODE,
                        )
                    ),
                    "text": chunk_text,
                    "site_url": str(
                        wert_oder_leer(
                            row,
                            "site_url",
                        )
                    ),
                    "file_url": str(
                        wert_oder_leer(
                            row,
                            "file_url",
                        )
                    ),
                    "publicbody": str(
                        wert_oder_leer(
                            row,
                            "publicbody",
                        )
                    ),
                    "foirequest": str(
                        wert_oder_leer(
                            row,
                            "foirequest",
                        )
                    ),
                    "uid": str(
                        wert_oder_leer(
                            row,
                            "uid",
                        )
                    ),
                    "source_text_path": str(
                        text_pfad.resolve()
                    ),
                    "clean_text_path": str(
                        bereinigter_text_pfad.resolve()
                    ),
                    "created_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                }

                alle_chunks.append(
                    chunk_eintrag
                )


            status_eintrag["num_chunks"] = len(
                dokument_chunks
            )

            status_eintrag["chunking_status"] = (
                "completed"
            )

            status_eintrag["processed_at"] = (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )

            alle_dokument_status.append(
                status_eintrag
            )

            erfolgreiche_dokumente += 1

            print(
                f"OK – {len(dokument_chunks)} Chunks"
            )


        except Exception as fehler:

            status_eintrag["chunking_status"] = (
                "failed"
            )

            status_eintrag["chunking_error"] = (
                f"{type(fehler).__name__}: {fehler}"
            )

            status_eintrag["processed_at"] = (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )

            alle_dokument_status.append(
                status_eintrag
            )

            fehlgeschlagene_dokumente += 1

            print(
                f"FEHLER: {status_eintrag['chunking_error']}"
            )


        if laufende_nummer % SAVE_EVERY == 0:

            chunks_df, chunking_status_df = (
                ergebnisse_speichern(
                    chunks=alle_chunks,
                    dokument_status=alle_dokument_status,
                )
            )

            print(
                f"    Zwischenstand nach "
                f"{laufende_nummer} Dokumenten gespeichert."
            )


except KeyboardInterrupt:

    print()
    print(
        "Verarbeitung wurde manuell unterbrochen. "
        "Der bisherige Fortschritt wird gespeichert."
    )


finally:

    chunks_df, chunking_status_df = (
        ergebnisse_speichern(
            chunks=alle_chunks,
            dokument_status=alle_dokument_status,
        )
    )


# ============================================================
# 11. ERGEBNISSE BERECHNEN
# ============================================================

anzahl_chunks = len(
    chunks_df
)

durchschnitt_chunks = (
    anzahl_chunks / erfolgreiche_dokumente
    if erfolgreiche_dokumente > 0
    else 0
)

durchschnitt_zeichen = (
    chunks_df["character_count"].mean()
    if (
        not chunks_df.empty
        and "character_count" in chunks_df.columns
    )
    else 0
)

durchschnitt_woerter = (
    chunks_df["word_count"].mean()
    if (
        not chunks_df.empty
        and "word_count" in chunks_df.columns
    )
    else 0
)


# ============================================================
# 12. ABSCHLUSSSTATISTIK
# ============================================================

print("\n" + "=" * 80)
print("TEXTBEREINIGUNG UND CHUNKING ABGESCHLOSSEN")
print("=" * 80)

print("\nVerarbeitete Dokumente insgesamt:")
print(f"{len(alle_dokument_status):,}")

print("\nErfolgreich verarbeitete Dokumente:")
print(f"{erfolgreiche_dokumente:,}")

print("\nLeere Dokumente:")
print(f"{leere_dokumente:,}")

print("\nFehlgeschlagene Dokumente:")
print(f"{fehlgeschlagene_dokumente:,}")

print("\nErzeugte Chunks:")
print(f"{anzahl_chunks:,}")

print("\nDurchschnittliche Chunks pro Dokument:")
print(f"{durchschnitt_chunks:.2f}")

print("\nDurchschnittliche Zeichen pro Chunk:")
print(f"{durchschnitt_zeichen:.2f}")

print("\nDurchschnittliche Wörter pro Chunk:")
print(f"{durchschnitt_woerter:.2f}")

print("\nBereinigte Volltexte:")
print(BEREINIGTE_TEXTORDNER)

print("\nChunk-Datei als JSONL:")
print(CHUNKS_JSONL)

print("\nChunk-Datei als CSV:")
print(CHUNKS_CSV)

print("\nDokumentstatus:")
print(DOKUMENT_STATUS_CSV)

print("\nFehlerliste:")
print(CHUNKING_FEHLER_CSV)


# ============================================================
# 13. STATUSÜBERSICHT ANZEIGEN
# ============================================================

status_uebersicht = (
    chunking_status_df[
        "chunking_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "chunking_status"
    )
    .reset_index(
        name="anzahl"
    )
)

print("\n" + "=" * 80)
print("STATUSÜBERSICHT")
print("=" * 80)

display(
    status_uebersicht
)


# ============================================================
# 14. BEISPIEL-CHUNKS ANZEIGEN
# ============================================================

print("\n" + "=" * 80)
print("BEISPIEL-CHUNKS")
print("=" * 80)

if chunks_df.empty:

    print(
        "\nEs wurden keine Chunks erzeugt."
    )

else:

    beispiel_spalten = [
        spalte
        for spalte in [
            "chunk_id",
            "document_id",
            "title",
            "page_start",
            "page_end",
            "character_count",
            "word_count",
            "text",
        ]
        if spalte in chunks_df.columns
    ]

    display(
        chunks_df[
            beispiel_spalten
        ].head(10)
    )